# 02 · Checks on the half-hourly panel

Verification behind `negprice/processing.py`. The panel itself is built by `make data` (`data/processed/panel.parquet`); this notebook only checks it. Each section says what is tested, why, and the result.

Column definitions: `references/data_dictionary.md`.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

import negprice.processing as proc
from negprice.config import INTERIM_DATA_DIR, RAW_DATA_DIR, REGIONS


def read(name: str) -> pd.DataFrame:
    return pd.read_parquet(INTERIM_DATA_DIR / f"{name}.parquet")


def wape(actual: pd.Series, forecast: pd.Series) -> float:
    """Weighted absolute percentage error: sum|actual - forecast| / sum|actual|."""
    return (actual - forecast).abs().sum() / actual.abs().sum()

## 1. Timestamp conventions

**Why:** six sources are joined on time. One source labelled by interval start, or carrying daylight saving, would shift every join by 30–60 minutes without raising any error.

**Expected:** naive timestamps, NEM time (UTC+10 all year), labelling the **end** of each interval.

### A. Every datetime column

In [ ]:
for f in sorted(INTERIM_DATA_DIR.glob("*.parquet")):
    df = pd.read_parquet(f)
    print(f"\n== {f.name}  ({len(df):,} rows)")
    for c in df.columns:
        s = df[c]
        if pd.api.types.is_datetime64_any_dtype(s):
            mins = sorted(s.dt.minute.dropna().unique())
            print(f"  {c:20} {str(s.dtype):22} {s.min()} → {s.max()}  minutes={mins[:12]}")

**Result:** all columns naive `datetime64[us]`. Dispatch runs 00:05 → 00:00 in 5-minute steps (interval end). Half-hourly interval columns start at 00:30 with minutes {0, 30}. Predispatch runs at 13:00 only; rooftop PV forecast versions are ≤ 12:30. Row counts reconcile exactly with 3 regions × 1,826 days (predispatch minus the Oct 2022 gap).

### B. Physical test: does solar noon sit where it should, all year?

The sun ignores clock conventions, so the centre of the rooftop PV curve is a solar-noon detector. With interval-end labels it should read solar noon + 15 min; daylight saving would show as a +1 h jump in Oct–Mar.

In [ ]:
pv_noon = read("rooftop_pv_actual")
pv_noon = pv_noon[pv_noon.TYPE == "MEASUREMENT"].dropna(subset=["POWER"])
t = pv_noon.INTERVAL_DATETIME
pv_noon["tod"] = t.dt.hour + t.dt.minute / 60
pv_noon["month"] = t.dt.month

# Power-weighted mean time of day = centre of the solar curve
pv_noon["w"] = pv_noon.tod * pv_noon.POWER
g = pv_noon.groupby(["REGIONID", "month"])[["w", "POWER"]].sum()
centre = (g.w / g.POWER).unstack("REGIONID")
print(centre.round(2))

**Result: passed.** Annual means NSW 12.16 h, VIC 12.57 h, SA 13.08 h: solar noon (from longitude) + ~15 min, and SA − NSW ≈ 55 min, VIC − NSW ≈ 25 min. No jump between Oct–Mar and Apr–Sep (no daylight saving). The ±15 min month-to-month wobble, highest in February and lowest in Oct–Nov, is the equation of time.

## 2. Dispatch price: regions, intervention, duplicates

**Why:** when AEMO intervenes in the market, dispatch runs twice. Only the pricing run (`INTERVENTION = 0`) sets the price participants are paid; keeping both rows would double-count those intervals.

In [ ]:
dp = read("dispatchprice")
print(dp.columns.tolist())
print(dp.REGIONID.value_counts())
print("duplicate keys:", dp.duplicated(["SETTLEMENTDATE", "REGIONID"]).sum())
print("missing RRP:", dp.RRP.isna().sum())
print(dp.groupby("REGIONID").RRP.describe().round(1))

**Result:** 525,888 rows per region (1,826 days × 288), no duplicates, no missing values. Minimum −$1,000 (market floor); maximum $20,300 (market price cap, FY2026).

`INTERVENTION` was dropped by `dataset.py`, so the check below goes back to the NEMOSIS cache. **The cache stores every column as text**, so it is converted first.

In [ ]:
files = sorted((RAW_DATA_DIR / "nemosis_cache").glob("*DISPATCHPRICE*.parquet"))
raw = pd.concat((pd.read_parquet(f) for f in files), ignore_index=True)
raw = raw[raw.REGIONID.isin(REGIONS)].assign(
    SETTLEMENTDATE=lambda d: pd.to_datetime(d.SETTLEMENTDATE, format="%Y/%m/%d %H:%M:%S").astype("datetime64[us]"),
    INTERVENTION=lambda d: pd.to_numeric(d.INTERVENTION),
    RRP=lambda d: pd.to_numeric(d.RRP),
)
print(raw.groupby(["REGIONID", "INTERVENTION"]).size())
print(raw.PRICE_STATUS.value_counts(dropna=False))

# Where an intervention run exists: does our file hold the pricing-run (INTERVENTION = 0) price?
iv = raw[raw.INTERVENTION == 1][["SETTLEMENTDATE", "REGIONID"]]
check = (iv.merge(raw[raw.INTERVENTION == 0], on=["SETTLEMENTDATE", "REGIONID"])
           .merge(dp, on=["SETTLEMENTDATE", "REGIONID"], suffixes=("_iv0", "_ours")))
print("intervention intervals in the study window:", len(check))
print("ours == pricing-run price:", (check.RRP_iv0 == check.RRP_ours).mean())

# Share of 5-min prices exactly $0
print((dp.RRP == 0).groupby(dp.REGIONID).mean().round(4))

**Result:** 3,487 intervention intervals per region; all prices `FIRM`. `dataset.py` keeps `INTERVENTION == 0` before dropping the column, so the match should be 1.0. The cache also holds Sep 2021 (NEMOSIS downloads whole months), which is outside the study window.

0.5–0.7% of 5-min prices are exactly $0, so the target "< 0" vs "≤ 0" changes base rates by under 1 point.

## 3. Resample to half-hours

**Rule:** the half-hour ending HH:30 holds the six 5-min intervals ending HH:05 … HH:30 (`ceil("30min")`).

### Price

In [ ]:
hh = proc.price_to_half_hourly(dp)
print(len(hh), "expected", 3 * 87_648)
print(hh.n_5min.value_counts())
print(hh.HH_END.min(), "→", hh.HH_END.max(), hh.HH_END.dtype)
print(hh.groupby("REGIONID").agg(
    mean_neg=("rrp_mean", lambda s: (s < 0).mean()),
    any_neg=("rrp_min", lambda s: (s < 0).mean()),
).round(3))

**Result:** 262,944 half-hours, all with exactly 6 intervals. Base rates match notebook 01, computed by an independent code path:

| Region | Half-hour mean < $0 | Any 5-min interval < $0 |
|---|---|---|
| SA | 25.2% | 29.9% |
| VIC | 21.3% | 26.1% |
| NSW | 6.7% | 10.9% |

### Demand and supply (`DISPATCHREGIONSUM`)

In [ ]:
rs = read("dispatchregionsum")
print(rs.columns.tolist())
print(rs.isna().sum())

hr = proc.regionsum_to_half_hourly(rs)
print(len(hr), "| same keys as price:",
      hr[["REGIONID", "HH_END"]].equals(hh[["REGIONID", "HH_END"]]))
print(hr.groupby("REGIONID").TOTALDEMAND.describe().round(0))

In [ ]:
# Lowest demand per region: real events or glitches?
for r in REGIONS:
    print(hr[hr.REGIONID == r].nsmallest(5, "TOTALDEMAND")[["HH_END", "TOTALDEMAND"]])

**Result:** no missing values; same keys as price. The lowest demand values are smooth midday blocks on sunny, low-activity days: NSW Sunday 16 Feb 2025 (2,582 MW), VIC Saturday 27 Dec 2025, SA Christmas Day 2025 (−284 MW). Real, kept.

## 4. Rooftop PV actuals

**Leakage question:** if values were revised long after the interval, the archive holds the revised number, not the one known at the time.

In [ ]:
pva = proc.pv_actual_half_hourly(read("rooftop_pv_actual"))
print(len(pva), "rows | expected", 3 * 87_648 - 6)
print("empty values:", pva.pv_actual_mw.isna().sum())

lag = (pva.pv_actual_known_at - pva.HH_END).dt.total_seconds() / 3600
print(lag.describe().round(2))
print("share published > 1 h after interval:", (lag > 1).mean().round(4))
print("share > 24 h:", (lag > 24).mean().round(4))

**Result:** every value published 19–36 min after its interval, none later, so stored values are the values known at the time; PV actuals up to 13:00 on D-1 are a safe feature. 34 half-hours have no measurement (6 absent rows + empty values); they stay missing.

## 5. Master grid

Every (region, half-hour) that should exist. Each source is left-joined onto it, so gaps show up as missing values instead of silently missing rows.

In [ ]:
grid = proc.half_hour_grid()
print(len(grid), "expected 262944")
print(grid.HH_END.min(), "→", grid.HH_END.max(), grid.HH_END.dtype)
print(grid.groupby(["REGIONID", "DAY"]).size().value_counts())
print("days:", grid.DAY.nunique(), "| first/last:", grid.DAY.min().date(), grid.DAY.max().date())
print("keys match price:",
      grid.sort_values(["REGIONID", "HH_END"])[["REGIONID", "HH_END"]]
          .reset_index(drop=True).equals(hh[["REGIONID", "HH_END"]]))

**Result:** 262,944 rows; all 5,478 region-days (3 × 1,826) have exactly 48 half-hours.

## 6. Join all sources + leakage check

In [ ]:
panel = proc.build_panel(
    grid, hh, hr, pva,
    read("predispatch_price"), read("predispatch_regionsum"),
    read("rooftop_pv_forecast"), read("silo_weather"),
)
print(panel.shape)
print(panel.columns.tolist())
na = panel.isna().sum()
print(na[na > 0])

# Leakage: every forecast must be known by issue time (13:00 on D-1 = DAY - 11 h)
issue = panel.DAY - pd.Timedelta(hours=11)
for col in ["pd_known_at", "pvf_known_at"]:
    ok = panel[col].isna() | (panel[col] <= issue)
    print(col, "all ≤ issue time:", ok.all())

**Result:** 262,944 rows, no row multiplication (`validate=` on every merge). Missing values are exactly the known ones: 4,464 = 3 × 1,488 predispatch (Oct 2022 gap) and 34 PV actuals. Every forecast was known by 13:00 on D-1.

## 7. Quality checks: values wrong but present

Structure (duplicates, gaps, timezone) is already checked. These look for frozen sensors, broken forecast periods and physically impossible values.

In [ ]:
p = panel
hour = p.HH_END.dt.hour

# 1. Completeness of every half-hour
print("n_5min all 6:", (p.n_5min == 6).all())

# 2. Price floor, and max per financial year (the cap rises each July)
print("floor respected:", (p.rrp_min >= -1000).all())
fy = (p.HH_END - pd.Timedelta("30min")).dt.to_period("Q-JUN").dt.qyear
print(p.groupby(fy).rrp_mean.max().round(0))

# 3. PV physics: no negatives, 0 at night, forecast quantiles ordered
night = hour.isin([22, 23, 0, 1, 2, 3])
print("PV actual min:", p.pv_actual_mw.min(), "| max at night:", p.loc[night, "pv_actual_mw"].max())
ordered = (p.pvf_powerpoelow <= p.pvf_powerpoe50) & (p.pvf_powerpoe50 <= p.pvf_powerpoehigh)
print("PV forecast quantiles ordered:", ordered.mean().round(5))


# 4. Frozen values: longest run of identical consecutive values (half-hours)
def longest_run(s):
    return s.groupby((s != s.shift()).cumsum()).size().max()


for col in ["totaldemand", "pd_rrp", "pd_totaldemand", "pv_actual_mw"]:
    s = p[col] if col != "pv_actual_mw" else p[col].where(hour.between(10, 15))   # PV: daytime only
    print(col, p.assign(s=s).dropna(subset=["s"]).groupby("REGIONID").s.apply(longest_run).to_dict())

# 5. Forecast vs actual by year: a jump flags a broken period
yr = p.DAY.dt.year
print(p.groupby([yr, "REGIONID"]).apply(
    lambda d: pd.Series({
        "demand_WAPE": wape(d.totaldemand, d.pd_totaldemand),
        "pv_WAPE": wape(d.pv_actual_mw, d.pvf_powerpoe50),
    })).unstack("REGIONID").round(3))

**Result:**

- Completeness, floor, PV physics and quantile order: all pass.
- Yearly price maxima equal each financial year's market price cap ($15,100 FY22 → $20,300 FY26).
- Forecast error stable across years (demand 2–8%, PV 9–14%): no broken year.
- **One finding:** `pd_rrp` frozen for up to 137 consecutive half-hours (NSW). Investigated below.

## 8. Problem periods

### A. Where are the frozen prices?

In [ ]:
def runs(df, col, min_len=12):
    """Runs of identical consecutive values, at least min_len half-hours long."""
    out = []
    for r, d in df.groupby("REGIONID"):
        g = (d[col] != d[col].shift()).cumsum()
        rl = d.groupby(g).agg(start=("HH_END", "first"), end=("HH_END", "last"),
                              value=(col, "first"), n=(col, "size"))
        out.append(rl[rl.n >= min_len].assign(REGIONID=r))
    return pd.concat(out).sort_values("n", ascending=False).reset_index(drop=True)


print(runs(p, "pd_rrp").head(15).to_string())
print(runs(p, "rrp_mean").head(15).to_string())

**Result:** the long runs are June 2022: AEMO's forecast frozen at the $15,100 cap while actual prices sat at the **$300 administered price cap** (energy crisis, then market suspension). Shorter runs elsewhere (SA 11 Jan 2022 cap forecast, NSW 6 Apr 2022 at $299.99, VIC 15–16 Jan 2026 at −$2.70 overnight) are genuine market outcomes and are kept.

### B. Administered-price window (`admin_price` flag)

Flag set in `config.ADMIN_PRICE_DAYS`. Kept for training (demand, PV and weather were real); excluded from backtest scoring and the battery dollar figure.

In [ ]:
p = proc.add_admin_price_flag(panel)
print("flagged half-hours:", p.admin_price.sum(), f"({p.admin_price.mean():.2%})")

june = p[p.DAY.between("2022-06-01", "2022-06-30")]
for col, val in [("rrp_mean", 300), ("pd_rrp", 15100)]:
    print(f"\n{col} = {val}")
    print(june[june[col] == val].groupby("REGIONID").HH_END.agg(["min", "max"]).to_string())

edge = p[p.DAY.between("2022-06-08", "2022-06-28")]
print("\n", edge.groupby(["DAY", "REGIONID"]).rrp_mean.mean().unstack().round(0).to_string())

In [ ]:
# Cap-price forecasts outside the window: genuine scarcity calls, or a broken forecast?
cap = p[p.pd_rrp == 15100]
print(cap.groupby(cap.DAY.dt.to_period("M")).size().to_string())

jun = cap[cap.DAY.between("2022-06-01", "2022-06-30")]
print(jun.groupby(["DAY", "REGIONID"]).size().unstack(fill_value=0).to_string())

**Result: window = 13–24 Jun 2022** (1,728 half-hours, 0.66%).

- Every $300 half-hour falls inside it: first NSW 13 Jun 19:30, last NSW 23 Jun 01:30. 12 Jun and 25–28 Jun look like a normal (expensive) winter market.
- Inside the window, the forecast sits at the cap for 32–48 half-hours a day; outside it, at most 15 (evening peaks: genuine scarcity forecasts, kept).
- 24 Jun stays flagged: its forecast was made on 23 Jun, during the suspension.
- $15,100 was the cap only in FY2022, so the monthly counts don't detect cap forecasts in later years; the run check in section 7 does, and found none.

### C. Dec 2024 rooftop PV forecast outage

Target days 15–17 Dec 2024 used a PV forecast 5–53 h old. Did AEMO's predispatch degrade too?

In [ ]:
dec = p[p.DAY.between("2024-12-08", "2024-12-22")]
err = dec.groupby(["DAY", "REGIONID"]).apply(lambda d: pd.Series({
    "pv_WAPE": wape(d.pv_actual_mw, d.pvf_powerpoe50),
    "dem_WAPE": wape(d.totaldemand, d.pd_totaldemand),
    "price_MAE": (d.rrp_mean - d.pd_rrp).abs().mean(),
})).unstack("REGIONID").round(2)
print(err.to_string())

**Result: no flag.** PV, demand and price errors on 15–17 Dec sit within the range of the surrounding normal days (cloudy days produce similar PV errors). Prices were real market outcomes and only an input was stale, which is what a production model will face during any AEMO outage.

Price error in dollars is dominated by spikes (NSW 13 Dec: $3,772).